In [1]:
import requests
import folium
import geopandas as gpd
from shapely.geometry import shape
from pyproj import CRS
import json
import os

Memanggil API

In [2]:
urls = {"resiko_banjir":'https://geoserver.mapid.io/layers_new/get_layer?api_key=96182a49207a421f974977b4cc184c61&layer_id=698e035302ca69c9817a2538&project_id=698dcfe49e209fa19435595f',
        "tingkat_polusi":'https://geoserver.mapid.io/layers_new/get_layer?api_key=96182a49207a421f974977b4cc184c61&layer_id=6992bc67e12474627a9f0e3e&project_id=698dcfe49e209fa19435595f',
        "kepadatan_penduduk":'https://geoserver.mapid.io/layers_new/get_layer?api_key=96182a49207a421f974977b4cc184c61&layer_id=69929f4d9e209fa194ce1af9&project_id=698dcfe49e209fa19435595f',
        "LUAS RTH":'https://geoserver.mapid.io/layers_new/get_layer?api_key=96182a49207a421f974977b4cc184c61&layer_id=6992a4289e209fa194d035ff&project_id=698dcfe49e209fa19435595f'
        }

Mengkonversi data API menjadi GeoDataFrame

In [10]:
import geopandas as gpd
import requests  # WAJIB ADA

def get_gdf_from_api(url):
    response = requests.get(url)  # pakai requests
    if response.status_code == 200:
        geojson = response.json()
        gdf = gpd.GeoDataFrame.from_features(geojson['features'])
        print(f"Berhasil Mengambil Data Dari {url.split('&layer_id=')[1][:10]}...")
        return gdf
    else:
        print(f"Gagal mengambil data dari {url}")
        return gpd.GeoDataFrame()

gdfs = {}

for key, url in urls.items():
    gdfs[key] = get_gdf_from_api(url)


Berhasil Mengambil Data Dari 698e035302...
Berhasil Mengambil Data Dari 6992bc67e1...
Berhasil Mengambil Data Dari 69929f4d9e...
Berhasil Mengambil Data Dari 6992a4289e...


Pemeriksaan & Penyesuaian Data

In [12]:
from pyproj import CRS
for key, gdf in gdfs.items():
    for col in gdf.columns :
        if gdf[col].dtype == '0':
            gdf[col] = gdf[col].fillna('Tidak Diketahui')
        else:
            gdf[col] = gdf[col].fillna(0)
    if gdf.crs is None or gdf.crs.to_epsg() != 4326:
        gdf.set_crs(epsg=4326, inplace=True)
        gdfs[key] = gdf
print("Semua Data sudah dicek dan disesuaikan")

Semua Data sudah dicek dan disesuaikan


visualisasi data setiap variabel

In [14]:
import folium

for key, gdf in gdfs.items():
    m = folium.Map(location=[-6.167, 106.758], zoom_start=12)
    folium.GeoJson(gdf).add_to(m)
    folium.LayerControl().add_to(m)
    display(m)
    print(f"Peta untuk variabel: {key}")


Peta untuk variabel: resiko_banjir


Peta untuk variabel: tingkat_polusi


Peta untuk variabel: kepadatan_penduduk


Peta untuk variabel: LUAS RTH
